# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chromium via Playwright (downloads its own browser — no system Chrome needed)
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [13]:
# Install Playwright + its own bundled Chromium (does NOT use system Chrome)
!pip install -q playwright nest_asyncio pandas tqdm
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [ ]:
import re
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
IGNORE_PATTERNS = ['noreply', 'users.noreply.github.com', 'github.com', 'githubusercontent']


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in IGNORE_PATTERNS:
        if pattern in email_lower:
            return False
    return True


async def start_browser(github_cookie=None):
    """Launch headless Chromium and return (pw, browser, context)."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    if github_cookie:
        context = await browser.new_context()
        await context.add_cookies([{
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        }])
        print('Browser started with GitHub session!')
    else:
        context = await browser.new_context()
        print('Browser started (anonymous — profile emails will be hidden)')

    return pw, browser, context


async def stop_browser(pw, browser):
    """Clean up browser and playwright."""
    try:
        await browser.close()
    except Exception:
        pass
    try:
        await pw.stop()
    except Exception:
        pass


async def scrape_email_from_profile(page, username):
    """Visit GitHub profile and extract email from page text."""
    try:
        await page.goto(f'https://github.com/{username}', wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(2000)

        body_text = await page.inner_text('body')

        emails = EMAIL_RE.findall(body_text)
        for email in emails:
            if is_valid_email(email):
                return email

        source = await page.content()
        mailto_matches = re.findall(r'mailto:([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', source)
        for email in mailto_matches:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


async def scrape_email_from_commits(page, username):
    """Get email from user's commit .patch files."""
    try:
        await page.goto(f'https://github.com/{username}?tab=repositories&type=source',
                         wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(1000)

        repo_elements = await page.query_selector_all('a[itemprop="name codeRepository"]')
        repo_names = []
        for el in repo_elements[:3]:
            name = await el.inner_text()
            repo_names.append(name.strip())

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                await page.goto(
                    f'https://github.com/{username}/{repo_name}/commits?author={username}',
                    wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1000)

                commit_links = await page.query_selector_all(
                    f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = await commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = (await commit_link.get_attribute('aria-label')) or ''
                    text = (await commit_link.inner_text()) or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    if href.startswith('/'):
                        href = 'https://github.com' + href

                    await page.goto(href + '.patch', timeout=15000)
                    await page.wait_for_timeout(1000)

                    page_text = await page.content()

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


async def scrape_email(page, username):
    """Try profile first, then commits."""
    email = await scrape_email_from_profile(page, username)
    if email:
        return email
    return await scrape_email_from_commits(page, username)


async def worker(worker_id, context, queue, results, pbar):
    """Worker that opens its own page and pulls usernames from the queue."""
    page = await context.new_page()
    try:
        while True:
            try:
                username = queue.get_nowait()
            except asyncio.QueueEmpty:
                break

            email = await scrape_email(page, username)
            results.append({'username': username, 'email': email})

            if email:
                print(f'  [W{worker_id}] ✓ {username} -> {email}')
            else:
                print(f'  [W{worker_id}] ✗ {username} -> not found')

            pbar.update(1)
            await page.wait_for_timeout(1000)  # be polite
    finally:
        await page.close()


print('Functions loaded. Ready to scrape.')

In [ ]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
swiderdnb
Chuanyin1202
samchenku
zd87pl
MarkLo127
xiezhifeng83
LingXuanTech
HamaFx
jimmy00415
raftersvk
rachnapandya
quyip8818
rajesh-chawla
quanliangliu
qq173681019
QiuMike
qchen34
QAMichaelPeng
Python-Z
pycaster
pruthviraj-chavan
PriteshPatel3
Premkumar09
prathanbomb
pkrishnath
peterwangz5
selvakumarEsra
SeaLeee
sdk451
SCzhJ
sap-anan
sangfansh
Sanfinity
sanchitsehgal8
samuelTyh
samad74a
sadofriod
sabotajj
rybzhw
rwforest
runwezh
RohiV04
Rocketbull
robertf99
rminchev1
rjaskirat
riskycheng
RinZ27
rickey-c
richzw
richtt02
realones
rdylimited
raramika
raptoravis
rangeryu
randomradio
rajeshthangaraj1
mulkymalikuldhrs
MrTheUniverse
Mr-Jack-Tung
mows21
Morrowind-Xie
mishra-aayush-work
MintTrader
Mintalix
mingl2000
mikko-ms-salonen
MikkelScheike
mikejohanson
mihailnica10
Meng-V
menbatisiunissart
MaxCYCHEN
maweis1981
Maurest0
matsebas
martinffx
markthien
marconildo
Mannafee
manav-vc
Malm
malekjaroslav
makoc0704
makk9
MaiKuraki
M00M00M00
LUOWY-26
PeterWang723
PerpetualWealth
paulpham157
Patricks15
patel-jeel92
PartyDolphin
parithoshp
panzhenye
PandaLehre
orapeasant
omar-gomake
OliverPanda
Objective-Collapse
obekt
oalotaik
oabdelmaksoud
nzuykov
norman-glad
Noorah233
nonomal
niyoh120
nikhil-nakhate
nidao003
NEXIO-GLOBAL
nepalisagun
NeoRosis
nberk2
MysticLiu
MysticalDawn
myh-st
MUmarJ
xypisces
xy7365527-lang
xwinwin
xupeng9
xuhaixing
xm-evanguo
xinleexin
xingdi1990
xiezhifeng883-ctrl
xiaoydl
XIAOJING-sin
xiaoCarl
xctest
xcellentbird
xblaster
xaxy55
Xavier-J-07
wzyyyqwzyyyq-maker
wook3024
wonkothesanest
wong813
Won-Ryeol
wo6x5a
wingyuenlabs
williscliao
whrit
whatqiu
weihong-su
wangziran-up
wangrenren611
Wang-cheng-123
WALL-E
zzy1927us-debug
zxc88645
ZuyangYu
zizhaof
zibo-wang
zhenglee1984
zhangmaosen
zhangchenhaobest
zeuswuwuwuwu
Zelray
zcxGGmu
zaynfuture
yysu
yybrother989
yuyangchen23
yushukk
yulinzhang96
yuhuichen
yuanchuzhiyi
ysnlly
yptse123
youking5688
yogee11
Yoengcode
ylzhang
yinjg1997
yimmy23
ydtan123
yazhou-zyz
YaakovHatam
y24hao123
xzYue
SuperYung
sunjoonkim
sunaina88
student59
stephenhe
Stefan824
startrekor
stanbird08
splumber
spacecomputer
SouthernIslands
soon95
sooahleex
sofanaja44
snab-xin
slavayosome
skywolf123
skalingclouds
Simonsaycheez
sidhunt
shudonglin
shuang950516
shiyush-git
shinchen03
ShevaShen
shepherd001
shenyisy
ShensiCeciliaLi
shengzhc
ShawnZ95
shantanushinde99
Sh1n
vuduclong0309
voodoo12345
vittoriop17
vishalbelsare
vijaygill
victor1589007281
velugubantla
Undertaker-afk
twmmason
tuwenxin1996-sudo
treasuraid
TracyHe
tomjod
tinypoint
TienYuWu
tianhm
thsvc
Thor-Coders-Inc
thinhnotes
thingthinker
thierryhzz
thickhair
the-ayyi
thainh11
tgraves83
textboy
TechnoHong
taobo0310
tankdonut
tanaer
tailgunner7777
t0my0ung
dickgibbons
dici-dev
devtrack
dev-indra
derrikzhang
DataLynnDev
daskabe
darrenxuan
Darle19
darkfriend77
DandiWong
cyberstormdotmu
cwjcw
ctfbindsec
Crimsonyx412
Count20
Coriolanus-Min
Connor-Shen
Colin579M
CloudEngineHub
clhcc
Cincinnatus-wxp3
Cian0Projects
ChrisGe4
chongli-uw
chirag-sharma
chinnsenn
chenyuan99
chaofanat
ChangranXU
chaien0609
cemreaytekin
kimifdw
FENGYYFF
feng-sh
fangchenli
ethz42
ethan-y-cheung
ertasenes
ericwang1124
Elena245
elegansn2
EKUL-Skywalker
EAniwa
eaglehhha
dulumao
ducga1998
dtarkent2-sys
dsx1986
DShaurin
drlappies
drjs07
Drenban
DrEden33773
DragonBtc93
dragon-entropy
dongyang-wu
DoneHome
DonaldChan608
domonic18
dma9527
dkuc-ingot
Dimmiditutto
digitalgym
andreakone
anandmuthu-vnix
amjadjibon
Amanzgr8
AlwaysFyo
allanhung
Alfred-Lau
alexwei12
alexcwyu
alcoat
alanhsun
aladinw
Akshaypatil15
Aitous
aiinpocket
ai-toolbox-hub
ahmedRabieMohamed
afanty2021
aeriosk
adrielp
adonis1491
adalard
abdallahsn
89jobrien
7897ewrr
4mophy
3DAlgoLab
3123958139
17lai
1376524890
0xthienngan
00make
ccc12138
cassgo
Camusama
buhu2k
brandonnmartinsj
bmigette
BioInfo
Bing4Ever
billrain
BiliKingdom
bhardwajRahul
BerthalonLucas
bensonmaxai
Ben-Liao
beiaihu
Baozhi888
ba5bo
B3-404
azaj01
Ax1an
Authcult
atbox-zz
aslldab1
aryeko
Artesania-Honuras
aroundble
arnjblancer
AristotleLai
arhow
AnthonySouthavilay
anishmantri
Andy-UT
khlin216
Kherrisan
khanhct
KforKelvin
kevinyankai
kevin0629
keemsisi
keeker765
kccchiu
kazuma-424
kaushik-yadav
kapilthakare-cyberpunk
Kamspol
KamisAyaka
Kaltsit-is-my-wife
kaattz
jyingjie
Joshuajxy
JoseBarreiros
jonnyquan
jolchmo
jjyn0215
jiaqizhang-zjq
jiangge
JiangFeng07
jiangbingo
jianfeng-personal
jeffreychuuu
jecketyuan-ux
jchenDevops
jburkey28
luohy15
lunaxoniichan
LuckieCat
louisfghbvc
longjing618
llmsc-security
LkxPro
Ljx-007
liyueyi
liuxuehai
liukunda
linquanzhi
LilachFarhi
likawa3b
leonwooster
LeonAI-DO-YLCS
LeoHuang4153
LeoAlija
leedstyh
layzhi
Laxmi884
LamLarryyyy
LakersLi
LabinatorSolutions
l1andisLiang
kp-forks
kooooichi24
koblick
knight099
klugben
KittenCN
kinwong-ds
hedeqiang
HCYT
haoshao
hanyu008-sys
hanryang0106
Hamiedamr
hamgor
halalquant
ha5h6r000wn
gymmic
guoz14
grandgen
godnight10061
gnoparus
gnarayan1
Geoseanlee
gejifeng
Gaber-Youssef
g4duck
fynnjuranek
fuyaozhao1018
fred-vu
franpb14
franklinlyj24
FrankCCCCC
fooyou-io
flowKKo
fisherwei
fireraintech
fgrfn
ferangarita01
FengZi-lv
javasoup
Jasonsey
james-deng
jacobp00
jacksu4
JackBeStrong
ivanleekk
Inupedia
inspire12
Insider77Circle
ilay-chen
iforgotmywallet
IdealDestructor
Ibrahim-Alnutayfi
iamsk
hyai-cn
Hy-ou
hxzrx
hwting1
huangjialin1118
HomuraCatMadoka
Hoder-zyf
hkt999
hitesh091
himyjan
Hill-Pham
hileamlakB
Hewei603
henryhu12
hemangjoshi37a
hellowutong
heisenyu
duncan60
aichiyu94
WeishuaiGu
jrayatt
liucy0417
HarveyTei
myqzero
stonerjurand
nextify2025
1917173927
vonhatcuong
JacksonTu
dox-box
hhljs
fuchsblau
goldkim92
CGRobinCNA
ihuangweiwei
CoryKrol
wo6x5a
nhz-io
Flytan
XBear0112
mono88
crazyhedgehogy
lemarco95
alex621
beaupranisaa
hsrho
Megane14916
1480046824
djv56666
Cookerjin
dasganni
david781110
victor777-coder
Lovoe
Kimihiro11
1139065556-rgb
hede317
cccaliy
RShahData
minebin
AnandInguva
Umbertovalachi
guyguhiohiuhiu
c001estb0y
agisota
yueking123
Kira-Pgr
xiandong79
wanglitongtong
langyanduan
demonzhao42-jpg
16Tesla
Jazz-251
htpbackup
gitssie
d1Cap1024
fwrite0920
binaryzipper
RayNCode
INGHRTY
cbaird-1337
ahmedxsamrat
sati2013
rickieplin
myownelixir2
cherizz00
ASeed223
antonioshadji
wanddream
xiedc1227-bot
zjltomicq
xucailiang
BigtoC
TaurusZz
hertzhzzz
huidark
UncleM1314
jialehu123
ChihayaAnon4750
Tullix95
Neon-Jing
Abolian2002
Make-magic
EonsClark
SpongeBobIsTheKingInTheNorth
tangfeihuang
IanBurger
EansonLee
shashankrajput000333-cloud
fivemoons
yiyiluo39-dev
alexmihalache
felixscode
T-wenhao
lierizhuoxin
mbcheng-8
soyking
lbaoyuan180-sys
opopligher1996
zhangqiang8878
Chitanda60
SiriusWy
zolicat
CamilleFelicity22
ruiwang20010702
phelma
yy0011-ui
zbw332
MortenQuant
PJerry168
JJXiangJiaoJun
AstaTus
cammclain
Radon333
iwanghy
Kakarot-zz
LoInTW
mwm333-design
FengZi-lv
YangWenxu
Weitong0513
Bencool
chitsanfei
bikumallapriyatham
jokillerftw
dinghaijie
parzival010
Carl-Huang
zpj23
zkaiWu
wincher-ruan
cjquery
marcosgabbardo
yingbaihu
benderninakitten69
seanmckdev
Cangning1
vvv-prog
andreafinotti
nfbs2000
olahelgedagsrud
CiprianSavoiu
adfhug435
VadimSmirnov213
DissidentNRV
RamblinGambler
blyt
jonasbaum06-art
skoobo
jovarela
Zed-777
XogZ3
researchase
SirOsborn
YeLuo45
zj276-commits
Pomelochen
arunkp89
bigcat26
ripotala2024
nisjit1505
jiangxiaopidezhanghao
victor-zzh
FlackoTurbo
ndiy
nlbutts
kwisser
Eileenxj
pjshy
edwinfrelancer-ops
bytelooper
zxy0253
lucasskk
Drknt00
qwe547
cgycorey
LarsKarbach
njylll
lt9
GunziS
mlboy
lynne-spark
xclly
flenczewski
gglive
ppea
alexleung0808-beep
Woody69-hunter
Annie5300
yanxxyy
nfsarch33
shawfen
xojuholic
scanhand
lqp518
Grigsuv
mhtnaim1
butzmeister
Owen-Qin
brandondees
IvanWang97
penyiqian
syx86
NeuralBlitz
southzyzy
wangchongyangdev-png
vinberm
quinnzheng68
MuXing9527
ilondo7jonathan-a11y
aniket4206
chuangzhidan
jac697
hlx2026
jmp2626
Alexlz2538
summer-pursuit
mahmoudsallem
WillamJackson
SamiINReciept
yuchiu503
Dongsheng600
eguan88
mattohan567
lisaaaaaaaaaaaaaaa
agussilva88
HenningBeyer
zenjing1024
chahakshahcs5
bmkrkolli
karanshergill
ProgrammPRO-33
DamienMarill
coldlikedecember
Kirrito-k423
estarter
ThinhPhan
shreeshjha
frankpi
Teeeeen
javasemeee
Leon555
john22122020-ship-it
teolzr
heroicmars-spec
mozhongzhou
a851445115
dexterpuru
alfajr666
rocke2020
alfalfac
kafka-snowflake
lyenliang
jxiaox
c8kmhjxgdb-lab
robin525
NowYouSeeMe-404
marionikola
starman-tech
xiaohei366
lehung2018
Meeweston
minhluudinh
archyqx
Alizestl
Auser6789
Respat30
kaihuaybj
JSantisteban96
finekewei
yulinzhang96
liyu916
PurpleStarCB
Ruiss1
mrhalyang
JLXIA
abalpay
wooosang
XiaoxiamiDongfang
fengjiantu-rgb
Imnondeersty
tianyuanqike
Jalaa532
yuvaraj119
way1122
Nokijai
huangyinxin
kobesu2018
wang6668888
jilliankk
c3msu
KdsTory
donotwarry
DaKonKon
zxzvsdcj
3ltrashpanda
coffeeguy999
Devmagic9
wangbinlbl
sahaavi
phoneee
lklimek
JasonWongLY
ezerfernandes
wade183
StarrYMSkY
lan041221
jigarmehta24
maosuyun2009
JasonWayne
zota10-ai
ckju1806
rprp
LAJOSR18
AzureDragon-Z
Activer007
Sergemoraru
Piyush-Agarwal20
zyh1690
bailingnan
dahu33
wanghy6503
xzarti
nichwang88
Kpowered
yum08
skywolf123
biebg
lurunze
Ylsssq926
ricky77melo
quanqigu
greyireland
karl-sy
AliAlQahtani
wewewil69
galaxyxym
kelhosary-rgb
yuandeshoulian
yirenzui8
windangle
Tasselyy
yryc07
IfAaron
GermMC
Zohra20
SusannaShu
caomeilaoshu
shr00mie
TangWu5912
IsLand-x
yunxingwoo
kedongxuexi
Canahmetozguven
shanggangli
4858409
Nurdich
TimeCracker
rishi003
xczhusuda
bslevin87
CWK2025-coder
ChangerHe
VietHieu95
jyothir07
pedro-pscunha
pamito1
oiiDawn
sambenito
qiz029
Fmick
w0rxbend
haishenghuo
RagulRakarot
chengchew0204
lailoo
MTJS-lf
limaoru
LeonardoBerti00
LiberJe
izm20
bugsbunny88
Surufan-cyber
alphadame999
JirenShen
mdj11j
ly91888-source
DhavalThkkar
Aurora03-god
841916140
Lam03tn
Nelayah
ycbai
Yuumi0221
ButlerSebastian
JaysonYJ
shipmints
0xthienngan
dankaperepelkin-creator
iGmR74
chriscrcodes
JinghuangLiu
gofunz
zelmanhenry5-ship-it
khooni-bhediya
Sandyuelin
lemonphoenix9
jiazewang13-alt
Hikohikoyan
cogitoergosumsw
zhoupeng6d
z775794057
CaboRU
s7monk
gityuantao
ikram98ai
ayush108108
enricozhang
liximin
szucielgp
saleh3636
rothbergguan-commits
Zin0703
crossover123
yojeajie
Kainiko943
twinkle-26
taddeusb90
joshuachen9
meijianwei8
nightelforc
buihuynhhung
hwhenry2023-droid
samwancd
0robin0
wukong1995
herdingSheepBug
leo0123456
liugetongban
zhangfaliang
mattnicomn
PrisonBreak2017
rivirside
windssea
DerisKoi
mimickn
messiahwww
willconry
camera88765
Inupedia
natea
jeffery-jhj
starlink-awaken
ericloiewl
hkievet
Yuge-225
manab1991-sudo
johndavekern-del
mrqli
joeshu578
wsm13143025
meaning-0f-life
hueta
wchen02
yevzh1
projectaqiu
MPKharche
tmpeters
ytliu1985
vibeyclaw
yongdono
clearoy
fms231
jinboy198
sprogii
vsairaghav
incipienstation
ZentyeZ
jchen91
TechAtlasDev
JPEDROPS092
ivyruiwang
futureinhands
bodphi
financeexpertfr
zojyjg
yaswhar
Fenglin24
Flamewaker
boss-mao
NdettoMbalu
liangwp001
songbae-medility
horlorlahdeh
Litre-WU
CharryWu
wjzhao
T4mako
John2024Mo
Hoohchaos
shen-shen
Whztever
willy89726015-netizen
TongoGuo
passion2021
rohitporeddy
leomonwei
veraaaaaLiu
helpaccident
Haibo114Luo
S-W-Laird
hjg-cell
AurelieKH2026
shakedaskayo
7Hu0v0
mtchen2016
YangZ0225
chanhz
OhmMyVolts
ayush3298
deepfates
klein-baru
liqi6919
Chaosqing
napwolf
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

In [ ]:
# Run the scraper — 5 parallel workers (5 browser tabs)
WORKERS = 5

pw, browser, context = await start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []

queue = asyncio.Queue()
for username in usernames:
    queue.put_nowait(username)

try:
    with tqdm(total=len(usernames), desc='Scraping emails') as pbar:
        tasks = [worker(i, context, queue, results, pbar) for i in range(WORKERS)]
        await asyncio.gather(*tasks)
finally:
    await stop_browser(pw, browser)

found_count = sum(1 for r in results if r['email'])
print(f'\nDone! Found {found_count}/{len(usernames)} emails')

In [17]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')

Total: 56
With email: 30
Without email: 26



,username,email
0,3rdAI-admin,None
1,ArkayaVenture,admin@arkayaventure.co.uk
2,Kaairofelipe,None
3,cpiprint,tcarson@cpiprint.com
4,cpiprint,tcarson@cpiprint.com
5,cpiprint,tcarson@cpiprint.com
6,cpiprint,tcarson@cpiprint.com
7,shreyasgm,shreyas.gm61@gmail.com
8,asfakahamedc,None
9,jsairdrop1,None


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>